# Détection RORS



In [ ]:
#chargement des modules
import pandas as pd
import datetime
import os
import json
from io import StringIO
import numpy as np
import plotly.express as px
import re

In [ ]:
import s3fs #pour connecter au bucket

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



On récupère le fichier avec les adresses produit à partir du notebook `get_geographical_date`.


In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_author_affil_by_publi.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep=",")


In [ ]:
df0["row_id"] = df0.id +"_"+df0.index.astype(str)
df01 = df0[["id","full_name","row_id","rors"]].copy() # tableau pour prendre uniquement les rors
df01["ror"] = df01.rors.str.split("|")
df_rors = df01.explode("ror").reset_index().drop(columns=["index"])
df_rors0 = df_rors.loc[df_rors.ror =="no ror"]
df_rors1 = df_rors.loc[df_rors.ror !="no ror"]
list_affil_without_ror = [x for x in df_rors0.row_id.unique()]
len(list_affil_without_ror)



In [ ]:
file_path = BUCKET_OUT + "/" + "Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.csv"
with fs.open(file_path) as csvin:
    df1 = pd.read_csv(csvin, sep =";")



## Retrouver les informations géographiques à partir du ROR

L'objectif est d'utiliser l'api de ROR pour récupérer des données géographiques à partir de l'identifiation ROR des institutions. Le problème est que toutes les institutions recencées dans le BSO n'ont pas de ROR. L'analyse des données montre qu'on dispose du ROR uniquement pour les établissement français.

On utilise la librairie requests pour interroger l'API

In [ ]:
df01 = df0[["id","rors"]]
df01["rors"] = df01.rors.str.split("|")
df_ror = df01.explode("rors")
df_ror

In [ ]:
import requests
from requests.exceptions import HTTPError
import urllib.parse
import tqdm

In [ ]:
ROR_API_ENDPOINT = "https://api.dev.ror.org/organizations"


params = {"query":"1m84wm78"} # le nombre de ligne. Par défaut, l'api affiche les 30 première ligne et le maximum fixé est de 10000.

In [ ]:
query_response = requests.get(ROR_API_ENDPOINT + '/' + "01m84wm78").json()

On affiche la structure des données

In [ ]:
#query_response

In [ ]:
for n, x in enumerate(query_response["names"]):
    if 'ror_display' in x["types"]:
        print(x)

Nous voulons récupérer les informations contenues dans "locations" :

```
locations': [{'geonames_details': {'continent_code': 'EU',
    'continent_name': 'Europe',
    'country_code': 'FR',
    'country_name': 'France',
    'country_subdivision_code': 'BRE',
    'country_subdivision_name': 'Brittany',
    'lat': 48.11198,
    'lng': -1.67429,
    'name': 'Rennes'},

```

In [ ]:
dict_ror = {}
list_row = []
for ror in tqdm.tqdm(df_ror.rors.unique(), total = df_ror.rors.nunique()):
    dict_row ={}
    rorid = ror.split("/")[-1]
    #print(rorid)
    dict_ror[ror] = rorid
    dict_row["rors"] = ror
    data = requests.get(ROR_API_ENDPOINT + '/' + rorid).json()

    for n, x in enumerate(data["names"]):
        if 'ror_display' in x["types"]:
            dict_row["nom"] = x["value"]
    locations_data = data["locations"][0]["geonames_details"]
    for i in locations_data:
        dict_row[i] = locations_data[i]
        if i == "name":
            dict_row["city"] = locations_data[i]
    dict_row["geonames_id"] = data["locations"][0]["geonames_id"]
    list_row.append(dict_row)


        
    
    
    

In [ ]:
df_inst = pd.DataFrame.from_dict(list_row)

In [ ]:
df_inst.country_name.value_counts()
df_inst

In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_geographical_info_french_affiliation.csv", "w") as file_out:
    df_inst.to_csv(file_out, sep=",", index= False)

In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_publication_with_french_affiliation.csv", "w") as file_out:
    df_ror.merge(df_inst, on = ["rors"], how = "left").drop_duplicates().to_csv(file_out, sep=",", index= False)


## Retrouver le ror à partir du nom des universités

Dans les données enrichies, le ror est présent uniquement pour les établissement français. On va essayer de récupérer le ror à partir de l'adresse des structures (lorsqu'elle est indiquée).
Pour cela, nous allons utilisés plus informations et plusieurs technique. L'objectif est de récupérer le nomn de l'établissement et la ville.

Lorsqu'il y a une adresse dans les données du bso (variable nommée `address`), nous l'utilisons en priorité. S'il n'y en a pas ou que les informations contenues ne permettent de retrouver des indications géographiques, on utilise l'affiliation retrouvée via Crossref.


**Pour certaines lignes, le texte présent dans les colonne "struc_address" et "address" correspondent en fait soit au titre soit au résumé de la plublication.**


Concernant les techniques, on utilise d'abord les regex, puis des outils de reconnaissance d'entité nommée (Spacy)




In [ ]:
file_path = BUCKET_OUT + "/" + "Data_bso/outputs/enriched_data/geolocation/2026-03-11_affil_crossref.csv"
with fs.open(file_path) as csvin:
    dfcr = pd.read_csv(csvin, sep =",")


dfcr1 = dfcr.loc[dfcr.affiliation!="no affil"]

df01 = df0.merge(dfcr1[["id","affiliation"]], on = ["id"], how = "left").merge(df1[["id","title"]], on = ["id"], how = "left")#.reset_index()


Le dataframe df_adfr contient une ligne par adresse pour chaque publication (si un une publication contient plusieurs adresses, il y aura plusieurs lignes). À partir de ces adresses (texte non structuré), nous allons récupérer les infos sur les établissements à l'aide de regex. On recherche plusieurs informations :

- Le nom des universités (Université Paris 8)
- La présence d'UMR, EA, UMS

In [ ]:
df02 = df01[["id","struc_address","affiliation","title"]]
df02["address"] = df02.struc_address.str.split("|")
df_ad = df02[['id','title','address']].explode("address").drop_duplicates().dropna(subset="address").reset_index().drop(columns=["index"])

#On isole les adresses françaises
df_adfr = df_ad.loc[(df_ad.address.str.lower().str.contains("france|paris"))].reset_index().drop(columns=["index"])

df_adfr

La colonne `rank_id` correspond à la position de l'adresse. On fusionnera ensuite les résultats en prenant en compte l'̀`id` et le `rank_id` 

In [ ]:
df_adfr["rank_id"] = df_adfr.index.astype(str)
df_adfr["name_umr"] = df_adfr.apply(lambda row:"|".join(set([x.replace(" ","") for x in re.findall(r"(?:umr|ums|ea).\d+\b", str(row.address).lower())])),1)

df_adfr.loc[df_adfr.name_umr.str.contains("ums")]

In [ ]:
df_adfr["address"] = df_adfr.apply(lambda row: re.sub("Paris VIII", "Paris 8", str(row.address)),1)
df_adfr["address"] = df_adfr.apply(lambda row: re.sub("Paris XIII", "Paris 13", str(row.address)),1)
df_adfr["address"] = df_adfr.apply(lambda row: re.sub("Paris X", "Paris Nanterre", str(row.address)),1)

In [ ]:
df_frwithoutumr = df_adfr.loc[df_adfr.name_umr==""]

In [ ]:
!python -m spacy download fr_core_news_md

In [ ]:
import spacy

In [ ]:
regex_match = {}

for index, row in df_adfr.iterrows():
    #print("######", row.address)
    address = re.sub(r"'|’", " ", str(row.address))
    split_address = set(re.sub(r"\s\-\s|;", ",",address.lower()).split(","))
    lower_address = ", ".join(split_address)
    regex = r"(?:univ\w*|collège|institut|école|school)[\s\w-]*\b"
    regex2=r'\b[\s\w-]+(?:univ\w*|collège|institut|école|school)\b'
    find_univ = re.findall(regex, lower_address)
    find_univ2 = [x.strip() for x in re.findall(regex2, lower_address) if x not in find_univ]

    list_univ = set(find_univ + find_univ2)
    if len(list_univ) > 0 :
        regex_match[row.rank_id]= "|".join(list_univ)
    else:
        pass
regex_match
df_adfr["etab"] = df_adfr.rank_id.map(regex_match.get)

Après l'application des regex, on construit un dataframe avec une ligne par établissement : il peut y avoir plusieurs établissements dans une adresse.

In [ ]:
t_etab = df_adfr[["id", "rank_id","etab"]]
t_etab["etab"] = t_etab.etab.str.split("|")
t_etab1 = t_etab.loc[~t_etab.etab.isna()].explode("etab").drop_duplicates().reset_index().drop(columns=["index"])
list_etab = [x for x in t_etab1.etab.unique()]
print(len(list_etab))


t_etab1

    

On va ensuite récupérer le ror des établissements listés dans `t_etab1` en utilisant les données ouvertes par le ministère de l'enseignement supérieur.

In [ ]:
file_path = BUCKET_OUT + "/" + "Data_bso/referentiels/fr-esr-principaux-etablissements-enseignement-superieur.csv"
with fs.open(file_path) as csvin:
    ref = pd.read_csv(csvin, sep =";")


In [ ]:

cleaned_list_etab = [re.sub(r'\s+', ' ', re.sub(r"d'|l'|\bl\b|\bd\b|\bde\b|\bdes\b", " ", x)) for x in list_etab]


dict_etab = {}

for index, row in ref.iterrows():
    univ = re.sub(r"d'|l'|\bde\b|\bdes\b", " ",row.uo_lib.lower())
    cleaned_univ = re.sub(r'\s+', ' ', univ)
    #cleaned_list_etab = [re.sub(r'\s*', ' ', re.sub(r"d'|l'|\bde\b|\bdes\b", " ", x)) for x in list_etab]
    find_etab = [x for x in cleaned_list_etab if re.match(cleaned_univ, x)]
    for etab in find_etab:
        dict_etab[etab]= row.uo_lib

dict_etab
t_etab1["cleaned_etab"] = t_etab1.apply(lambda row: re.sub(r'\s+', ' ', re.sub(r"d'|l'|\bl\b|\bd\b|\bde\b|\bdes\b", " ", str(row.etab))),1)
t_etab1["norm_name"] = t_etab1.cleaned_etab.map(dict_etab.get)
t_etab1["id_rors"] = t_etab1.norm_name.map(dict(zip(ref.uo_lib, ref.identifiant_ror)).get)
#t_etab1["rors"] = "https://ror.org/" + t_etab1.id_rors.astype(str)
t_etab1

In [ ]:
list_etab1 = list(set([x for x in t_etab1.etab.loc[t_etab1.id_rors.isna()]]))
print(len(list_etab1))
dict_ror2 = {}
for m, etab in tqdm.tqdm(enumerate(list_etab1), total = len(list_etab1)):
    row_id = m
    l_apostrophe = re.sub(r"\sl\s", " l'", etab)
    cleaned_etab = re.sub(r"\sd\s", " l'", l_apostrophe)
    if len(cleaned_etab.split(" ")) > 2:
        response = requests.get(ROR_API_ENDPOINT + '?' + f'query="{cleaned_etab}"' + "&filter=country.country_code:fr").json()
        if response["number_of_results"] > 1 :
            for n, x in enumerate(response["items"]):
                if 'education' in x["types"] :
                    datas = response["items"][n]
                    rors = datas["id"]
        elif response["number_of_results"] == 1 :
            datas = response["items"][0]
            rors = datas["id"]
        else:
             rors = "not found"
    else:
        rors = "not found"
    dict_ror2[etab] = rors
    #print(cleaned_etab, rors)

**Nécessaire de vérifier les rors**

In [ ]:
dict_ror2

In [ ]:
t_etab2 = t_etab1.loc[~t_etab1.id_rors.isna()]
t_etab3 = t_etab1.loc[t_etab1.id_rors.isna()]
t_etab2["rors"] = "https://ror.org/" + t_etab2.id_rors.astype(str)
t_etab3["rors"] = t_etab3.etab.map(dict_ror2.get)
t_etab3.loc[t_etab3.rors!="not found"].drop_duplicates("cleaned_etab")


In [ ]:
df_allrors = pd.concat([t_etab2[["id","rank_id","rors"]], t_etab3[["id","rank_id","rors"]]])

#df_adfr["rors"] = df_adfr.rank_id.map(dict(zip(df_allrors.rank_id, df_allrors.rors)))
df_umr = df_adfr.loc[df_adfr.name_umr!=""]
df_umr["umr"] = df_umr.name_umr.str.split("|")
df_umr1 = df_umr[["id","rank_id","umr"]].explode("umr").reset_index().drop(columns=["index"])
df_umr1["umr_id"] = df_umr1.index.astype(str)
df_umr1

In [ ]:
list_umr = list(set([x for x in df_umr1.umr]))
print(len(list_umr))
dict_ror3 = {}
for m, umr in tqdm.tqdm(enumerate(list_umr), total = len(list_umr)):
    label_umr = umr.replace("umr","umr ").replace("ea","ea ").replace("ums", "ums ")
    
    response = requests.get(ROR_API_ENDPOINT + '?' + f'query="{label_umr}"' + "&filter=country.country_code:fr").json()
    if response["number_of_results"] > 1 :
        for n, x in enumerate(response["items"]):
            if 'education' in x["types"] :
                datas = response["items"][n]
                rors = datas["id"]
    elif response["number_of_results"] == 1 :
        datas = response["items"][0]
        rors = datas["id"]
    else:
         rors = "not found"
    dict_ror3[umr] = rors
    #print(cleaned_etab, rors)

In [ ]:
file_path = BUCKET_OUT + "/" + "/Data_bso/referentiels/fr-esr-repertoire-national-structures-recherche-historique-annuel.csv"
with fs.open(file_path) as csvin:
    umr_histo = pd.read_csv(csvin, sep =";")
    

file_path = BUCKET_OUT + "/" + "/Data_bso/referentiels/fr-esr-structures-recherche-publiques-actives.csv"
with fs.open(file_path) as csvin:
    umr = pd.read_csv(csvin, sep =";")
    
umr1 = umr.loc[~umr.label_numero.isna()]
umr1.loc[umr1.label_numero.str.contains("EA 4178")]
umr_histo1 = umr_histo.loc[~umr_histo["Label numéro"].isna()]
umr_histo1.loc[(umr_histo1["Label numéro"].str.contains("UMR 8608"))&(umr_histo1["Année"]==umr_histo1["Année"].max())].iloc[0]

In [ ]:
file_path = BUCKET_OUT + "/" + "/Data_bso/referentiels/fr-esr-repertoire-national-structures-recherche.csv"
with fs.open(file_path) as csvin:
    rnsr = pd.read_csv(csvin, sep =";")

rnsr.libelle.loc[rnsr.numero_national_de_structure=="202023524M"].iloc[0]

In [ ]:
print(len([x for x in dict_ror3 if dict_ror3[x]=="not found"]))


            


In [ ]:
t_etab3

In [ ]:
dict_libel_umr = {}
for umr in tqdm.tqdm(dict_ror3, total= len(dict_ror3)):
    if dict_ror3[umr] == "not found":
        #print("######", umr)
        label_umr = umr.replace("umr_","umr").replace("umr","umr ").replace("ea","ea ").replace("ums", "ums ").replace("  ", " ")
        try:
            libelle = umr1.libelle.loc[umr1.label_numero.str.lower().str.contains(label_umr)].iloc[0]
            commune = umr1.commune.loc[umr1.label_numero.str.lower().str.contains(label_umr)].iloc[0]
        except:
            libelle = "unknown"
            commune = "unknown"
        if libelle != "unknown":
            dict_libel_umr[umr] = libelle.lower()
            libelle= libelle.replace("&","and")
            #print(libelle)
            response = requests.get(ROR_API_ENDPOINT + '?' + f'query="{libelle.lower()}"' + "&filter=country.country_code:fr").json()
            if response["number_of_results"] > 1 :Université Mohammed V de Rabat
                for n, x in enumerate(response["items"]):
                    if 'education' in x["types"] :
                        datas = response["items"][n]
                        rors = datas["id"]
                        dict_ror3[umr] = rors
            elif response["number_of_results"] == 1 :
                datas = response["items"][0]
                rors = datas["id"]
                dict_ror3[umr] = rors
            else:
                rors = "not found"
                dict_ror3[umr] = rors


In [ ]:
dict_libel_umr

In [ ]:
print(len([x for x in dict_ror3 if dict_ror3[x]!="not found"]))
df_umr1["rors"] = df_umr1.umr.map(dict_ror3.get)
df_umr1["norm_name"] = df_umr1.umr.map(dict_libel_umr.get)
df_umr2 = df_umr1.loc[df_umr1.rors!="not found"]


df_allrors = pd.concat([t_etab2[["id","rank_id","rors","etab","norm_name"]], t_etab3[["id","rank_id","rors","etab","norm_name"]]]).drop_duplicates()
d = df_allrors[["rors","etab","norm_name"]].merge(df_umr2[["rors", "umr", "norm_name"]], on = ["rors"], how= "left")
d_univ = d[["rors", "etab", "norm_name_x"]].loc[(d.rors!="not found") & (d.umr.isna())].drop_duplicates().sort_values("norm_name_x")
d_umr = d[["rors", "etab", "norm_name_y","umr"]].loc[(d.rors!="not found") & ~(d.umr.isna())].drop_duplicates()
d_univ.loc[~d_univ.norm_name_x.isna()]

In [ ]:
d_univ["pattern"] = d_univ.apply(lambda row: re.sub(r"\-|\s|'", "", str(row.norm_name_x).lower()),1)
d_univ["string"] = d_univ.apply(lambda row: re.sub(r"\-|'|\s", "", str(row.etab).lower()), 1)
d_univ_pattern = d_univ[["rors","pattern"]].loc[~d_univ.norm_name_x.isna()]
d_univ_string = d_univ[["rors","etab","string"]].loc[d_univ.norm_name_x.isna()]
d_univ_string.merge(d_univ_pattern, on = ["rors"], how = "left")
d_univ2 = pd.concat([d_univ[["rors","etab","string","pattern"]], d_univ_string.merge(d_univ_pattern, on = ["rors"], how = "left")]).drop_duplicates()
d_pattern = d_univ2[["rors","pattern"]].loc[~d_univ2.pattern.isna()].drop_duplicates()
dict_pattern_univ = {}

for index, row in d_pattern.iterrows():
    if isinstance(row.pattern, float):
        pass
    elif row.pattern == 'none':
        pass
    else:
        dict_pattern_univ[row.rors] = row.pattern
print(len(dict_pattern_univ))

d_univ_without_pattern = d_univ2[["rors","string","pattern"]].loc[d_univ2.pattern.isna()].drop_duplicates()

for index, row in d_univ_without_pattern.iterrows():
    if isinstance(row.string, float):
        pass
    elif row.pattern == 'none':
        pass
    else:
        dict_pattern_univ[row.rors] = row.string
print(len(dict_pattern_univ))
d_univ["pattern"]= d_univ.rors.map(dict_pattern_univ)

dict_variante = {}
for rors in dict_pattern_univ:
    
    dtmp = d_univ.loc[d_univ.rors==rors]
    list_variante= [x for x in dtmp.etab.unique()]
    dict_variante[rors] = "|".join(list_variante)

dict_variante
df_inst["variantes"] = df_inst.rors.map(dict_variante.get)

In [ ]:
list_ror = [x for x in df_inst.rors.unique()]
list_ror2 = [x for x in dict_variante if x not in list_ror]
dict_ror = {}
list_row = []
for ror in tqdm.tqdm(dict_variante, total = len(dict_variante)):
    dict_row ={}
    rorid = ror.split("/")[-1]
    #print(rorid)
    dict_ror[ror] = rorid
    dict_row["rors"] = ror
    data = requests.get(ROR_API_ENDPOINT + '/' + rorid).json()

    for n, x in enumerate(data["names"]):
        if 'ror_display' in x["types"]:
            dict_row["nom"] = x["value"]
    locations_data = data["locations"][0]["geonames_details"]
    for i in locations_data:
        dict_row[i] = locations_data[i]
        if i == "name":
            dict_row["city"] = locations_data[i]
    dict_row["geonames_id"] = data["locations"][0]["geonames_id"]
    list_row.append(dict_row)


In [ ]:
df_inst2 = pd.DataFrame.from_dict(list_row)
df_inst2["variantes"] = df_inst2.rors.map(dict_variante.get)
df_inst = pd.concat([df_inst, df_inst2]).drop_duplicates()
df_inst

In [ ]:
list_ror = [x for x in df_inst.rors.unique()]
list_ror2 = [x for x in d_umr.rors.unique() if x not in list_ror]
print(len(list_ror), len(list_ror2))

dict_variante = {}
dict_variante_umr = {}
for rors in d_umr.rors.unique():
    
    dtmp = d_umr.loc[d_umr.rors==rors]
    list_variante= [x for x in dtmp.etab.unique()]
    variante_umr =  [x for x in dtmp.umr.unique()]
    dict_variante[rors] = "|".join(list_variante)
    dict_variante_umr[rors] = "|".join(variante_umr)

dict_ror = {}
list_row = []
for ror in tqdm.tqdm(list_ror2, total = len(list_ror2)):
    dict_row ={}
    rorid = ror.split("/")[-1]
    #print(rorid)
    dict_ror[ror] = rorid
    dict_row["rors"] = ror
    data = requests.get(ROR_API_ENDPOINT + '/' + rorid).json()

    for n, x in enumerate(data["names"]):
        if 'ror_display' in x["types"]:
            dict_row["nom"] = x["value"]
    locations_data = data["locations"][0]["geonames_details"]
    for i in locations_data:
        dict_row[i] = locations_data[i]
        if i == "name":
            dict_row["city"] = locations_data[i]
    dict_row["geonames_id"] = data["locations"][0]["geonames_id"]
    list_row.append(dict_row)
df_inst3 = pd.DataFrame.from_dict(list_row)
df_inst3["variantes"] = df_inst3.rors.map(dict_variante.get)
df_inst = pd.concat([df_inst, df_inst3]).drop_duplicates()
df_inst["umr_variante"] = df_inst.rors.map(dict_variante_umr.get)
df_inst

In [ ]:
d_univ0 = d[["rors", "etab", "norm_name_x"]].loc[(d.rors=="not found")].drop_duplicates().sort_values("norm_name_x")
d_univ00 = d_univ0.loc[~d_univ0.norm_name_x.isna()] #sans rors, avec norm_name_x
d_univ01 = d_univ0.loc[d_univ0.norm_name_x.isna()] #sans rors, sans norm_name_x
d_univ00["pattern"] = d_univ00.apply(lambda row: re.sub(r"\-|\s|'", "", str(row.norm_name_x).lower()),1)#
d_univ00["string"] = d_univ00.apply(lambda row: re.sub(r"\-|'|\s", "", str(row.etab).lower()), 1)
d_univ00["rors"]= "https://ror.org/01ahyrz84"
df_inst.loc[df_inst.rors=="https://ror.org/01ahyrz84"]

dict_variante = {}
dict_variante_umr = {}
for rors in d_univ00.rors.unique():
    
    dtmp = d_univ00.loc[d_univ00.rors==rors]
    list_variante= [x for x in dtmp.etab.unique()]
    dict_variante[rors] = "|".join(list_variante)

dict_ror = {}
list_row = []
for ror in tqdm.tqdm(d_univ00.rors.unique(), total = d_univ00.rors.nunique()):
    dict_row ={}
    rorid = ror.split("/")[-1]
    #print(rorid)
    dict_ror[ror] = rorid
    dict_row["rors"] = ror
    data = requests.get(ROR_API_ENDPOINT + '/' + rorid).json()

    for n, x in enumerate(data["names"]):
        if 'ror_display' in x["types"]:
            dict_row["nom"] = x["value"]
    locations_data = data["locations"][0]["geonames_details"]
    for i in locations_data:
        dict_row[i] = locations_data[i]
        if i == "name":
            dict_row["city"] = locations_data[i]
    dict_row["geonames_id"] = data["locations"][0]["geonames_id"]
    list_row.append(dict_row)
df_inst4 = pd.DataFrame.from_dict(list_row)
df_inst4["variantes"] = df_inst4.rors.map(dict_variante.get)
df_inst = pd.concat([df_inst, df_inst4]).drop_duplicates().reset_index()
df_inst

In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_geographical_info_french_affiliation.csv", "w") as file_out:
    df_inst.to_csv(file_out, sep=",", index= False)

In [ ]:
df_inst.loc[df_inst.nom.str.contains("Évry")]

In [ ]:
d_univ01
d_univ01["string"] = d_univ01.apply(lambda row: re.sub(r"\-|'|\s", "", str(row.etab).lower()), 1)
list_pattern=[x for x in d_univ.pattern.unique()]

dict_pattern = {}
for index, row in d_univ01.iterrows():
    for p in list_pattern:
        if re.search(p, row.string):
            dict_pattern[row.string] = p
        else:
            pass

d_univ01["pattern"]= d_univ01.string.map(dict_pattern.get)
d_univ01.loc[~d_univ01.pattern.isna()]

In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_address_to_verify.csv", "w") as file_out:
    d_univ01.sort_values("pattern").to_csv(file_out, sep=",", index= False)

In [ ]:
#récupéré sur Le Chat via la demande suivante : how can i detect unstructured address in text

nlp = spacy.load("fr_core_news_md")

for index, row in df_frwithoutumr.head(20).iterrows():
    print("######", row.address)
    doc = nlp(row.address)
    for ent in doc.ents:
        #print(ent.text)
        ent.
        if ent.label_ == "ORG" or ent.label_ == "LOC":
            print(ent.text)


# Les institutions non françaises

In [ ]:
df_nofr = df_ad.loc[~(df_ad.address.str.lower().str.contains("france|paris"))].reset_index().drop(columns=["index"])
df_nofr#.loc[~df_nofr.affiliation.isna()]

In [ ]:
for 

In [ ]:
key ='École doctorale Pratiques et théories du sens'
status_response = requests.get(ROR_API_ENDPOINT + '?' + f'query="{key}"').json()
status_response

In [ ]:
dict_ror2 = {}
for m, a in enumerate(df_ad2.address.unique()[0:10]):
    row_id = m
    a_split = re.sub(r"[;,\.]", ";", a).split(";")
    nom_univ = re.sub(r"\[.*\]", "", a_split[0]).strip().replace("&egrave;", "è").replace("&", "%26%"). lower()
    #print(nom_univ)
    #key ='École doctorale Pratiques et théories du sens'
    
    try:
        response = requests.get(ROR_API_ENDPOINT + '?' + f'query="{nom_univ}"').json()
        if response["number_of_results"] > 1 :
            for n, x in enumerate(response["items"]):
                if 'education' in x["types"] :
                    datas = response["items"][n]
                    rors = datas["id"]
        elif response["number_of_results"] == 1 :
            datas = response["items"][0]
            rors = datas["id"]
        else:
             rors = "not found"
        dict_ror2[a] = rors
    except:
        print(nom_univ)

    

In [ ]:
dict_ror2
#len(df_ad2)

In [ ]:
dict_ror2
df_ad2["ror"] = df_ad2.address.map(dict_ror2.get)
df_ad2